In [1]:
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
import os
import sys
file_path = os.getcwd()
project_root = os.path.dirname(file_path)
print(project_root)
# Add the project root to the sys.path
if project_root not in sys.path:
    sys.path.append(project_root)
from dotenv import load_dotenv

from langchain.agents import AgentExecutor
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.agents import create_tool_calling_agent
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.prompts import PromptTemplate
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings
from langchain_core.tools import tool
from pathlib import Path
from IPython.display import display, Image
from typing import Optional,Dict, List, Tuple 
import subprocess
import os
import sys
import shutil # 
import re
from datetime import datetime 
import os
from pathlib import Path
# Our Libraries
from crunch_llm.code.agents import *
from crunch_llm.code.teams_SciML_PINN import *
from crunch_llm.code.utils import *

/users/jdtoscan/.conda/envs/Agent_Hub/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/oscar/data/gk/jdtoscan/LLM


# Initialization

In [2]:
my_database_path = "/users/jdtoscan/data/jdtoscan/LLM/Documents/database_pinns"
my_database_path_guides="/users/jdtoscan/data/jdtoscan/LLM/Documents/database_manuals"

In [3]:
naming_agent = StructuredAgent(model='gpt-oss:20b', 
                     verbose=False,
                     system_prompt=naming_agent_system_prompt_PINN,
                    )
retriever_agent = Retriever_Agent(model='gpt-oss:20b', 
                     verbose=False,
                     system_prompt=Retriever_system_prompt_PINN,
                     db_path=my_database_path,
                     collection_name='PINNs_Codes',
                     k=5)
planner_agent = aRAGAgent(model='gemini-2.5-pro', 
                     verbose=False,
                     system_prompt=planner_agent_system_prompt_PINN,
                     model_provider='google',     
                    embeddings_provider='google', 
                    embeddings_model='models/text-embedding-004',
                     db_path=my_database_path_guides,
                     collection_name='PINNs_Manual_v1',
                     k=2)
patcher_agent = StructuredAgent(
                    model='gemini-2.5-flash',
                    model_provider='google',
                    system_prompt=patcher_agent_system_prompt_PINN,
                    verbose=False
                )

# patcher_agent_gpt = StructuredAgent(model='gpt-oss:120b', 
#                      verbose=False,
#                      system_prompt=patcher_agent_system_prompt_PINN,
#                     )


## User Query

In [4]:
user_question ="""
Lets solve the viscous burgers equation u_t+uu_x=nu u_{xx} with IC=-sin(pi x) the domain goes from 0 to 1 in time and from -1 to 1 in x; periodic boundary conditions.

The reference data to compute the testing errors has been saved on the available datasets folder with the name burgers.mat

We will be solving a forward problem so $\nu=1/(100*pi)$
"""

# Retrieval Team

In [5]:
max_iterations_all=5
file_content, filename, retrieval_approval, validator_feedback=retrieval_team(
                                                                        user_question,
                                                                        retriever_agent,
                                                                        max_iterations= 5)

--- Retrieval Iteration: 0 ---
Retriever Message:
[Allen_Cahn.py]

**Justification**
I've selected the `Allen_Cahn.py` template. This is our standard hybrid Adam+SSBroyden solver, which is the correct methodological choice for the user's 1D Burgers' equation forward problem.
✅ Match found: Retriever successfully retrieved 'Allen_Cahn.py'
✅ Retrieval loop approved file: Allen_Cahn.py


## Code Refinement

In [6]:
file_content=code_refinement_team(user_question=user_question,
                    initial_code=file_content,
                     error_report='Please remember to change the names',
                     filename=filename,
                     planner_agent=planner_agent,
                     patcher_agent=patcher_agent
                     )

--- Retrieving documents for query: 'Cell 7 PDE Residual purpose' ---
Cell 2 
 *   **Action:** Update the default equation name.
*   **Details:** To ensure logs and output directories are named correctly, find the line `parser.add_argument('--Equation', type=str, default='Allen_Cahn', help='Name of equation')` and replace `'Allen_Cahn'` with `'Burgers'`.
--- Applying deterministic line range patch ---
  - Processing change for lines 7-7: action='replace'
    - Replacing lines 7-7 with 1 new lines.
Cell 3 
 *   **Action:** Load the correct dataset for the Burgers' equation.
*   **Details:** The user has specified that the reference data is in `burgers.mat`. Find the line `data = scipy.io.loadmat(available_data_sets_path+'allen_cahn.mat')` and replace it with `data = scipy.io.loadmat(available_data_sets_path+'burgers.mat')`.
--- Applying deterministic line range patch ---
  - Processing change for lines 4-4: action='replace'
    - Replacing lines 4-4 with 1 new lines.
Cell 4 
 *   **Acti

## Filing Team

In [7]:
new_code_details='Test'
requires_name=True
project_dir,filename,final_filepath,new_code_details= filing_team_PINN(user_question,filename,file_content,naming_agent,requires_name=requires_name,new_code_details=new_code_details)
requires_name=False

Proposed name:Burgers_Periodic_SinPi\Question :
Lets solve the viscous burgers equation u_t+uu_x=nu u_{xx} with IC=-sin(pi x) the domain goes from 0 to 1 in time and from -1 to 1 in x; periodic boundary conditions.

The reference data to compute the testing errors has been saved on the available datasets folder with the name burgers.mat

We will be solving a forward problem so $
u=1/(100*pi)$


Created project directory: /users/jdtoscan/data/jdtoscan/LLM/Results/PINN_codes//Burgers_Periodic_SinPi/2025_10_31_2252/
Successfully saved code to: /users/jdtoscan/data/jdtoscan/LLM/Results/PINN_codes//Burgers_Periodic_SinPi/2025_10_31_2252/Burgers_Periodic_SinPi.py


In [9]:
success,log_file_name=run_code(project_dir,filename,final_filepath)

Attempting to run /users/jdtoscan/data/jdtoscan/LLM/Results/PINN_codes//Burgers_Periodic_SinPi/2025_10_31_2252/Burgers_Periodic_SinPi.py...

--- [Streaming Output silently to /users/jdtoscan/data/jdtoscan/LLM/Results/PINN_codes//Burgers_Periodic_SinPi/2025_10_31_2252/Burgers_Periodic_SinPi.py.out] ---
--- [End of Stream. Exit Code: 0] ---
✅ Successfully ran Burgers_Periodic_SinPi.py.
